# 03. Анализ графа

**User Story 3**: Кластеризация, метрики центральности, обнаружение shell-компаний и циклических платежей.

**Процесс**:
1. Загружаем отфильтрованный граф из pickle
2. Leiden кластеризация (CPM) с подбором gamma
3. Вычисление метрик центральности (PageRank, betweenness, clustering)
4. Классификация ролей узлов
5. Обнаружение shell-компаний
6. Обнаружение циклических платежей
7. Сводка по кластерам

**Предпосылки**: Запустите `02_graph_construction.ipynb`.

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.analysis import (
    run_leiden_clustering,
    compute_centrality,
    classify_node_roles,
    detect_shell_companies,
    detect_cycles,
    build_cluster_summary,
)

## Загрузка графа

In [ ]:
# Загружаем отфильтрованный граф из pickle (локальная ФС)
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
print(f'Loading graph from: {graph_path}')

with open(graph_path, 'rb') as f:
    G = pickle.load(f)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## Leiden кластеризация

In [ ]:
membership, best_gamma = run_leiden_clustering(G)

# Применяем к графу
for node, cluster_id in membership.items():
    if node in G:
        G.nodes[node]['cluster'] = cluster_id

n_clusters = len(set(v for v in membership.values() if v >= 0))
print(f'\nBest gamma: {best_gamma}')
print(f'Clusters found: {n_clusters}')

In [ ]:
# Распределение размеров кластеров
cluster_sizes = pd.Series(membership).value_counts().sort_index()
cluster_sizes = cluster_sizes[cluster_sizes.index >= 0]  # exclude unassigned
print(f'\nCluster size distribution:')
print(cluster_sizes.describe())

fig, ax = plt.subplots(figsize=(10, 4))
cluster_sizes.plot(kind='bar', ax=ax)
ax.set_xlabel('Cluster ID')
ax.set_ylabel('Members')
ax.set_title('Cluster Size Distribution')
plt.tight_layout()
plt.show()

## Метрики центральности

In [ ]:
metrics_df = compute_centrality(G)

# Применяем к графу
for node_id, row in metrics_df.iterrows():
    if node_id in G:
        G.nodes[node_id]['pagerank'] = row['pagerank']
        G.nodes[node_id]['betweenness'] = row['betweenness']

print('Top 10 by PageRank:')
top_pr = metrics_df.nlargest(10, 'pagerank')[['pagerank', 'betweenness', 'in_degree', 'out_degree', 'total_in_flow']]
display(top_pr)

print('\nTop 10 by Betweenness:')
top_bc = metrics_df.nlargest(10, 'betweenness')[['pagerank', 'betweenness', 'clustering_coef', 'flow_through_ratio']]
display(top_bc)

## Классификация ролей

In [ ]:
metrics_df = classify_node_roles(metrics_df, G)

# Применяем роли к графу
for node_id, row in metrics_df.iterrows():
    if node_id in G:
        G.nodes[node_id]['role'] = row['role']

print('Role distribution:')
display(metrics_df['role'].value_counts())

## Shell-компании

In [ ]:
shell_df = detect_shell_companies(metrics_df, G)

if len(shell_df) > 0:
    # Добавляем shell_score в основной metrics_df
    metrics_df['shell_score'] = 0.0
    for idx in shell_df.index:
        metrics_df.loc[idx, 'shell_score'] = shell_df.loc[idx, 'shell_score']
    
    print(f'Flagged shell companies: {len(shell_df)}')
    display_cols = ['pagerank', 'betweenness', 'clustering_coef', 'flow_through_ratio',
                    'has_salary_payments', 'shell_score', 'role']
    display(shell_df[[c for c in display_cols if c in shell_df.columns]])
else:
    metrics_df['shell_score'] = 0.0
    print('No shell companies flagged.')

## Циклические платежи

In [ ]:
cycles = detect_cycles(G)

if cycles:
    print(f'Cycles detected: {len(cycles)}')
    for i, cyc in enumerate(cycles[:10]):
        node_names = [G.nodes[n].get('name', str(n))[:20] for n in cyc['nodes']]
        print(f"  Cycle {i+1}: length={cyc['length']}, amount={cyc['total_amount']:,.0f}")
        print(f"    Nodes: {' → '.join(node_names)}")
else:
    print('No cycles detected.')

## Сводка по кластерам

In [ ]:
cluster_summary = build_cluster_summary(G, metrics_df, cycles)

if len(cluster_summary) > 0:
    display(cluster_summary)
else:
    print('No clusters to summarize.')

## Сохранение результатов

In [ ]:
# Сохраняем обновлённый граф с кластерами и метриками
graph_path = config.OUTPUT_FILTERED_GRAPH_PICKLE
with open(graph_path, 'wb') as f:
    pickle.dump(G, f)
print(f'Analyzed graph saved: {graph_path}')

# Метрики — Pandas Parquet (локальная ФС)
# Добавляем cluster_id к метрикам
metrics_df['cluster_id'] = metrics_df.index.map(lambda n: G.nodes[n].get('cluster', -1) if n in G else -1)
metrics_df.to_parquet(config.OUTPUT_GRAPH_METRICS)
print(f'Metrics saved: {config.OUTPUT_GRAPH_METRICS}')

if len(cluster_summary) > 0:
    cluster_summary.to_parquet(config.OUTPUT_CLUSTERS, index=False)
    print(f'Clusters saved: {config.OUTPUT_CLUSTERS}')

---

**Следующий шаг**: Откройте `04_visualization.ipynb` для интерактивной визуализации.